# Assignment 2: MovieLens Data Pipeline with Spark and Cassandra
**Course:** STQD6324_Data Management

**Student:** FENG XIAOWEI_P165401

**Environment Configuration:**
* Python Version: 2.7.5
* Apache Spark Version: 2.3.0
* Cassandra Version: 3.0.9
* Cluster Environment: HDP Sandbox (maria_dev)

## Phase 1: Environment Setup and Pre-requisites
Before executing the PySpark data pipeline code within this Jupyter Notebook, the following data ingestion tasks and database initialization steps must be performed manually within the virtual machine terminal (e.g., via PuTTY).

### 1. Ingesting Data into HDFS 
Execute the following Hadoop Distributed File System (HDFS) commands to create the designated directory structure and upload the raw MovieLens 100k dataset files:

```bash
hdfs dfs -mkdir -p /user/maria_dev/ml-100k/
hdfs dfs -put u.user /user/maria_dev/ml-100k/
hdfs dfs -put u.data /user/maria_dev/ml-100k/
hdfs dfs -put u.item /user/maria_dev/ml-100k/

### 2. Initializing Cassandra Keyspace and Tables 
Switch to the root account to start the Cassandra service and launch the cqlsh interactive shell.

### 3. Launching Jupyter Notebook with Cassandra Connector Dependency
To enable communication between Apache Spark and Cassandra, the Jupyter Notebook instance must be initialized by passing the coordinate for the Datastax Spark-Cassandra Connector:

PYSPARK_SUBMIT_ARGS='packages com.datastax.spark:spark-cassandra-connector_2.11:2.3.0 pyspark-shell' jupyter notebook ip=0.0.0.0

## Phase 2: Building the Data Pipeline
### Step 1: Import Required Libraries and Initialize SparkSession
In this step, we import the core PySpark components, the Row object, and built-in functions, while configuring Spark to connect to the local Cassandra instance (IP: 127.0.0.1).

In [ ]:
from pyspark.sql import SparkSession, Row
from pyspark.sql import functions as F

# Initialize SparkSession with Cassandra configuration
spark = SparkSession.builder \
    .appName("MovieLens_Cassandra_Pipeline") \
    .config("spark.cassandra.connection.host", "127.0.0.1") \
    .getOrCreate()

# Get SparkContext
sc = spark.sparkContext
print("SparkSession successfully initialized with Cassandra configuration.")

### Step 2: Load Data from HDFS into RDDs
Using `sc.textFile`, the raw user, rating, and movie dataset files are read from the HDFS paths configured in Phase 1 to create the initial RDD objects.

In [ ]:
# Load raw datasets from HDFS
user_raw_rdd = sc.textFile("hdfs:///user/maria_dev/ml-100k/u.user")
data_raw_rdd = sc.textFile("hdfs:///user/maria_dev/ml-100k/u.data")
item_raw_rdd = sc.textFile("hdfs:///user/maria_dev/ml-100k/u.item")

print("u.user RDD count: " + str(user_raw_rdd.count()))
print("u.data RDD count: " + str(data_raw_rdd.count()))
print("u.item RDD count: " + str(item_raw_rdd.count()))

### Step 3: Data Cleaning and DataFrame Transformation
According to the MovieLens dataset specifications:
* `u.user` and `u.item` are delimited by the pipe character (`|`).
* `u.data` is delimited by whitespace/tabs.

We use the `map` function to split the raw strings, perform explicit type casting (e.g., converting strings to `int` or `float`), and map them into `Row` objects. Finally, `spark.createDataFrame` is used to convert them into strongly-typed Spark DataFrames.

In [ ]:
# 1. Parse and transform user RDD -> DataFrame
user_parsed = user_raw_rdd.map(lambda line: line.split('|')) \
                          .map(lambda p: Row(user_id=int(p[0]), age=int(p[1]), gender=p[2], occupation=p[3], zip=p[4]))

# 2. Parse and transform rating RDD -> DataFrame
data_parsed = data_raw_rdd.map(lambda line: line.split()) \
                          .map(lambda p: Row(user_id=int(p[0]), movie_id=int(p[1]), rating=float(p[2]), timestamp=p[3]))

# 3. Parse and transform movie RDD -> DataFrame
genres = ["unknown", "Action", "Adventure", "Animation", "Childrens", "Comedy", "Crime", 
          "Documentary", "Drama", "Fantasy", "Film_Noir", "Horror", "Musical", "Mystery", 
          "Romance", "Sci_Fi", "Thriller", "War", "Western"]

def parse_movie_row(line):
    p = line.split('|')
    movie_id = int(p[0])
    title = p[1]
    row_dict = {"movie_id": movie_id, "title": title}
    for i, g_name in enumerate(genres):
        row_dict[g_name] = int(p[5 + i]) if len(p) > (5 + i) else 0
    return Row(**row_dict)

item_parsed = item_raw_rdd.map(parse_movie_row)

# Convert RDDs to Spark DataFrames
df_users = spark.createDataFrame(user_parsed)
df_ratings = spark.createDataFrame(data_parsed)
df_movies = spark.createDataFrame(item_parsed)

# Register DataFrames as temporary SQL views
df_users.createOrReplaceTempView("users")
df_ratings.createOrReplaceTempView("ratings")
df_movies.createOrReplaceTempView("movies")

print("DataFrame transformation and Spark SQL view registration completed.")

### Step 4: Execute Analytical Queries
#### Tasks i and ii: Calculate Average Movie Ratings and Identify Top 10 Movies
We use Spark SQL to aggregate individual user scores from the `ratings` view using a `GROUP BY` clause and calculate the average rating via the `AVG` function. The result is then joined with the `movies` view using a `JOIN` operation to retrieve the movie titles, sorted in descending order using `ORDER BY`, and limited to the top 10 highest-rated movies.

In [ ]:
# Execute aggregation, join, and sorting via Spark SQL
top_movies_df = spark.sql("""
    SELECT m.title, ROUND(AVG(r.rating), 2) as avg_rating
    FROM ratings r
    JOIN movies m ON r.movie_id = m.movie_id
    GROUP BY m.title
    ORDER BY avg_rating DESC, m.title ASC
""")

# Display only the top 10 records
top_movies_df.show(10, truncate=False)

**[Interpretation & Discussion]**:
The query output indicates that the top 10 movies all achieved a perfect average score of 5.0. However, in practical scenarios, this is typically caused by data sparsity, where certain niche or obscure movies received only a single 5-star rating from one user. Therefore, in real-world large-scale recommendation systems (such as Netflix or Amazon), it is necessary to introduce a minimum rating count threshold or apply a Bayesian Average smoothing mechanism to prevent these extreme outliers from skewing the analytical results.

#### Task iii: Identify Users with at Least 50 Ratings and Determine Their Favourite Genre
First, a Common Table Expression (CTE) is used to filter active users who have rated 50 or more movies in the `ratings` table. Next, these active users' rating records are joined with the movie genre feature matrix. Since the genres are stored horizontally across multiple columns, we use Spark SQL's built-in map and `explode` functions to unpivot these columns into rows. Finally, we aggregate the frequency of each genre per user and apply the `ROW_NUMBER()` window function to extract the most frequently rated (favourite) genre for each user.

In [ ]:
# Execute genre unpivoting and window-based ranking via Spark SQL
favorite_genre_df = spark.sql("""
    WITH active_users AS (
        SELECT user_id 
        FROM ratings 
        GROUP BY user_id 
        HAVING COUNT(movie_id) >= 50
    ),
    user_genres AS (
        SELECT r.user_id, explode(map(
            'Action', m.Action, 'Adventure', m.Adventure, 'Animation', m.Animation,
            'Childrens', m.Childrens, 'Comedy', m.Comedy, 'Crime', m.Crime,
            'Documentary', m.Documentary, 'Drama', m.Drama, 'Fantasy', m.Fantasy,
            'Film_Noir', m.Film_Noir, 'Horror', m.Horror, 'Musical', m.Musical,
            'Mystery', m.Mystery, 'Romance', m.Romance, 'Sci_Fi', m.Sci_Fi,
            'Thriller', m.Thriller, 'War', m.War, 'Western', m.Western
        )) as (genre, is_tagged)
        FROM ratings r
        JOIN movies m ON r.movie_id = m.movie_id
        WHERE r.user_id IN (SELECT user_id FROM active_users)
    ),
    genre_counts AS (
        SELECT user_id, genre, COUNT(*) as genre_rating_count
        FROM user_genres
        WHERE is_tagged = 1
        GROUP BY user_id, genre
    ),
    ranked_genres AS (
        SELECT user_id, genre, genre_rating_count,
               ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY genre_rating_count DESC) as rank
        FROM genre_counts
    )
    SELECT user_id, genre as favourite_genre, genre_rating_count
    FROM ranked_genres
    WHERE rank = 1
    ORDER BY user_id ASC
""")

# Display the computation results for the first 10 active users
favorite_genre_df.show(10)

**[Interpretation & Discussion]**:
Through this query, we can determine the specific interest profiles (e.g., Drama, Comedy, etc.) of our target consumers. By unpivoting and flattening the behavioral interaction matrix into explicit preference tags, we establish the baseline for big data User Profiling. With these defined genre preference metrics, the platform can deliver highly targeted, precision content recommendations tailored directly to these core active users.

#### Tasks iv & v: Filter Specific Demographic Segments
In these two tasks, we utilize relational filtering predicates (`WHERE`, `AND`, `BETWEEN`) against the `users` view to extract target user profiles based on specific demographic attributes:
* **Task iv**: Filters all young users under the age of 20.
* **Task v**: Filters mid-career scientists between the ages of 30 and 40.

In [ ]:
# Task iv: Filter users under the age of 20
young_users_df = spark.sql("SELECT user_id, age, gender, occupation FROM users WHERE age < 20")
print("Demographic Segment - Users Under 20 (Top 10 Records):")
young_users_df.show(10)

# Task v: Filter scientists between 30 and 40 years old
scientists_filtered_df = spark.sql("""
    SELECT user_id, age, gender, occupation 
    FROM users 
    WHERE occupation = 'scientist' AND age BETWEEN 30 AND 40
""")
print("Demographic Segment - Scientists Aged 30-40:")
scientists_filtered_df.show(10)

**[Interpretation & Discussion]**:
By leveraging Spark SQL's filtering mechanisms, we can achieve precise Market Segmentation. For instance, the younger demographic under 20 can be targeted with mainstream, animation, or action-heavy movies. Conversely, the "scientists aged 30-40" represents a highly educated, niche user group that can be targeted with specialized marketing campaigns and tailored recommendations, such as hard sci-fi films or high-brow documentaries.

### Step 5: Write Processed DataFrames to Cassandra Database
In this step, the processed DataFrame data residing in Spark's memory is persisted into the Cassandra database. Specifically, we write the cleaned user profiles and the high-rated movie ranking results calculated in Tasks i & ii into their respective Cassandra tables.

In [ ]:
# 1. Write df_users to the 'users' table in Cassandra
df_users.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("append") \
    .options(table="users", keyspace="movielens") \
    .save()
print("User profile data successfully written to Cassandra.")

# 2. Write top_movies_df analytical results to the 'top_movies' table in Cassandra (Tasks i & ii)
top_movies_df.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("append") \
    .options(table="top_movies", keyspace="movielens") \
    .save()
print("Top movies analytical results successfully written to Cassandra.")

# 3. Write favorite_genre_df analytical results to the 'user_favorite_genre' table in Cassandra (Task iii)
favorite_genre_df.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("append") \
    .options(table="user_favorite_genre", keyspace="movielens") \
    .save()
print("User favorite genre results successfully written to Cassandra.")

# 4. Write young_users_df segment data to the 'young_users' table in Cassandra (Task iv)
young_users_df.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("append") \
    .options(table="young_users", keyspace="movielens") \
    .save()
print("Young users segment data successfully written to Cassandra.")

# 5. Write scientists_filtered_df segment data to the 'middle_aged_scientists' table in Cassandra (Task v)
scientists_filtered_df.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("append") \
    .options(table="middle_aged_scientists", keyspace="movielens") \
    .save()
print("Middle-aged scientists segment data successfully written to Cassandra.")

### Step 6: Reload Data from Cassandra for Integrity Verification
To ensure that no data corruption or structural loss occurred during the NoSQL ingestion process, we implement a reverse data-flow read as the final step. Spark is used to reload the two recently persisted tables from Cassandra back into Spark DataFrames, followed by a data display to perform a closed-loop validation.

In [ ]:
# 1. Verify and read back the 'users' table
validate_users = spark.read \
    .format("org.apache.spark.sql.cassandra") \
    .options(table="users", keyspace="movielens") \
    .load()
print("[Verification Success] Sample of first 5 rows reloaded from Cassandra 'users' table:")
validate_users.show(5)

# 2. Verify and read back the 'top_movies' analytical results table (Tasks i & ii)
validate_top_movies = spark.read \
    .format("org.apache.spark.sql.cassandra") \
    .options(table="top_movies", keyspace="movielens") \
    .load()
print("[Verification Success] Top 10 movie ranking records reloaded and re-sorted from Cassandra:")
validate_top_movies.orderBy("avg_rating", ascending=False).show(10)

# 3. Verify and read back the 'user_favorite_genre' analytical results table (Task iii)
validate_fav_genre = spark.read \
    .format("org.apache.spark.sql.cassandra") \
    .options(table="user_favorite_genre", keyspace="movielens") \
    .load()
print("[Verification Success] Sample of user favorite genres reloaded from Cassandra:")
validate_fav_genre.show(5)

# 4. Verify and read back the 'young_users' demographic segment table (Task iv)
validate_young = spark.read \
    .format("org.apache.spark.sql.cassandra") \
    .options(table="young_users", keyspace="movielens") \
    .load()
print("[Verification Success] Sample of young users profile reloaded from Cassandra:")
validate_young.show(5)

# 5. Verify and read back the 'middle_aged_scientists' demographic segment table (Task v)
validate_scientists = spark.read \
    .format("org.apache.spark.sql.cassandra") \
    .options(table="middle_aged_scientists", keyspace="movielens") \
    .load()
print("[Verification Success] Sample of mid-career scientists profile reloaded from Cassandra:")
validate_scientists.show(5)